# 🧹 Notebook 02 — Data Preprocessing
**Healthcare RAG-Powered Medical Q&A Assistant**

**Goal:** Take the raw CSV and turn it into a clean, ready-to-use dataset.

The raw data has 3 key columns:
- `question` → the medical question (already clean)
- `context` → list of PubMed abstract snippets (already extracted)
- `long_answer` → the detailed answer

We will join the context list into a single string, rename columns, and save a new file called `pubmedqa_cleaned.csv`

## Step 1 — Import Libraries

In [1]:
import pandas as pd
import re
import os
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

print('Libraries imported successfully!')

Libraries imported successfully!


## Step 2 — Load the Raw Data

In [2]:
# Load the raw dataset
df = pd.read_csv('../data/raw/pubmedqa_raw.csv')

print('Number of rows:', len(df))
print('Columns:', df.columns.tolist())
print()

# Preview the first row
df.head(2)

Number of rows: 211269
Columns: ['pubid', 'question', 'context', 'long_answer', 'final_decision']



,pubid,question,context,long_answer,final_decision
0,25429730,Are group 2 innate lymphoid cells ( ILC2s ) in...,{'contexts': ['Chronic rhinosinusitis (CRS) is...,"As ILC2s are elevated in patients with CRSwNP,...",yes
1,25433161,Does vagus nerve contribute to the development...,{'contexts': ['Phosphatidylethanolamine N-meth...,Neuronal signals via the hepatic vagus nerve c...,yes


## Step 3 — Extract Question, Context, and Answer

The raw data uses the **qiaojin/PubMedQA** schema:
- `question` is already clean — we use it directly
- `context` is a **list of strings** — we join them into a single text block
- `long_answer` contains the detailed answer — we rename it to `answer`

In [3]:
# Use the preprocessor module which handles both schemas
# (handles context list joining, column renaming, + legacy backward compat)
import sys
import os
sys.path.append(os.path.abspath('..'))
from src.data.preprocessor import run_preprocessing_pipeline, print_pipeline_log

# Run the full extraction pipeline (handles both qiaojin & legacy schemas)
df, pipeline_log = run_preprocessing_pipeline(df)

print_pipeline_log(pipeline_log)
print()
print('Done! Here is a sample:')
print()
print('QUESTION:', df['question'][0])
print()
print('CONTEXT (first 200 chars):', df['context'][0][:200])
print()
print('ANSWER:', df['answer'][0])



📊 Preprocessing Pipeline Log
──────────────────────────────────────────────────
  Raw rows:              211,269
  After extraction:      211,269
  After text cleaning:   211,269
  Removed missing:       0
  After missing filter:  211,269
  Removed non-English:   28
  After English filter:  211,241
  Removed duplicates:    53
  Final clean rows:      211,188
──────────────────────────────────────────────────

Done! Here is a sample:

QUESTION: are group 2 innate lymphoid cells ( ilc2s ) increased in chronic rhinosinusitis with nasal polyps or eosinophilia?

CONTEXT (first 200 chars): chronic rhinosinusitis (crs) is a heterogeneous disease with an uncertain pathogenesis. group 2 innate lymphoid cells (ilc2s) represent a recently discovered cell population which has been implicated 

ANSWER: as ilc2s are elevated in patients with crswnp, they may drive nasal polyp formation in crs. ilc2s are also linked with high tissue and blood eosinophilia and have a potential role in the activation 

## Step 4 — Clean the Text

We remove things that are not useful for NLP:
- Extra spaces
- Special characters like HTML tags
- Very long spaces or newlines

In [4]:
def clean_text(text):
    # Remove extra spaces and newlines
    text = re.sub(r'\\s+', ' ', str(text))
    # Remove HTML tags if any
    text = re.sub(r'<[^>]+>', '', text)
    # Remove leading and trailing spaces
    text = text.strip()
    return text

df['question'] = df['question'].apply(clean_text)
df['context']  = df['context'].apply(clean_text)
df['answer']   = df['answer'].apply(clean_text)

print('Text cleaned!')
print('Sample cleaned question:', df['question'][0])

Text cleaned!
Sample cleaned question: are group 2 innate lymphoid cells ( ilc2s ) increased in chronic rhinosinusitis with nasal polyps or eosinophilia?


## Step 5 — Check for Missing Values

In [5]:
print('Missing values in each column:')
print(df.isnull().sum())
print()

# Remove any row that has no question or no answer
before = len(df)
df = df.dropna(subset=['question', 'answer'])
df = df.reset_index(drop=True)
after = len(df)

print(f'Removed {before - after} rows with missing values')
print(f'Remaining rows: {after:,}')

Missing values in each column:
question    0
context     0
answer      0
dtype: int64

Removed 0 rows with missing values
Remaining rows: 211,188


## Step 6 — Remove Duplicate Questions

In [6]:
duplicates_q = df[df.duplicated(subset=['question'])]
duplicates_c = df[df.duplicated(subset=['context'])]
duplicates_a = df[df.duplicated(subset=['answer'])]
duplicates_q

,question,context,answer
11048,does clotting factor concentrate given to prev...,the hallmark of severe hemophilia is recurrent...,there is strong evidence from randomised contr...
25732,does lumbar support for prevention and treatme...,lumbar supports are used in the treatment of l...,there is still a need for high quality randomi...
32395,do the effect of remin pro and mi paste plus o...,the growing demand for enhanced esthetic appea...,there was no difference between surface roughn...
39226,do patients with pathologically proven renal d...,"to determine if patients with pathological, me...",the presence of pathological renal disease in ...
56307,do neither antioxidants nor cox-2 inhibition p...,reflux-induced injury and oxidative stress res...,"in this model of reflux injury, antioxidants a..."
...,...,...,...
202853,does the effect of chewing khat leave on human...,chewing fresh leaves of the khat plant (catha ...,khat chewing did result in functional mood dis...
204254,do interventions for helping people adhere to ...,chronic venous ulcer healing is a complex clin...,it is unclear whether interventions designed t...
209176,do placental/umbilical cord blood-derived mese...,the potential of human mesenchymal stem cell-l...,the present results showed the potential effic...
209177,does chromium attenuate hepatic damage in a ra...,oxidative stress is involved in cholestasis-in...,the data indicate that chromium attenuates bdl...


In [7]:
before = len(df)
# df = df.drop_duplicates(subset=['question'])
df = df.drop_duplicates()
df = df.reset_index(drop=True)
after = len(df)

print(f'Removed {before - after} duplicate questions')
print(f'Remaining rows: {after:,}')

Removed 0 duplicate questions
Remaining rows: 211,188


## Step 7 — Final Check & Save

In [8]:
# Final summary
print('===== FINAL DATASET SUMMARY =====')
print(f'Total rows     : {len(df):,}')
print(f'Total columns  : {len(df.columns)}')
print(f'Missing values : {df.isnull().sum().sum()}')
print(f'Columns        : {df.columns.tolist()}')
print('=================================')

# Verify we have exactly the 3 expected columns
expected_cols = ['question', 'context', 'answer']
assert list(df.columns) == expected_cols, \
    f"Expected {expected_cols}, got {list(df.columns)}"

# Save the cleaned file
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/pubmedqa_cleaned.csv', index=False)

print()
print('✅ Saved to: data/processed/pubmedqa_cleaned.csv')

===== FINAL DATASET SUMMARY =====
Total rows     : 211,188
Total columns  : 3
Missing values : 0
Columns        : ['question', 'context', 'answer']

✅ Saved to: data/processed/pubmedqa_cleaned.csv


In [9]:
# ── Optional: sync processed data to HuggingFace Hub ──────────────────────
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

if not os.getenv("HF_TOKEN"):
    print("\n⚠️  HF_TOKEN not set — skipping HuggingFace upload.")
    print("   Add it to your .env file (see .env.example).")
else:
    import sys
    sys.path.insert(0, str(PROJECT_ROOT))
    from src.data.hub import upload_file
    print("\n📤 Syncing to HuggingFace...")
    upload_file("data/processed/pubmedqa_cleaned.csv", "processed/pubmedqa_cleaned.csv")


📤 Syncing to HuggingFace...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  ✅ Uploaded: data/processed/pubmedqa_cleaned.csv (349.5 MB)


## ✅ Done!

| Step | What we did |
|---|---|
| Extract | Got clean question, context, answer from raw columns |
| Clean | Removed extra spaces and HTML |
| Missing | Removed rows with no question or answer |
| Duplicates | Removed exact duplicate rows |

**Output:** `data/processed/pubmedqa_cleaned.csv` (3 columns: question, context, answer)

**➡️ Next: Open `03_category_labelling.ipynb`**